# 3.3 查詢與被查詢為什麼分開？

# 第 3 章：Attention，自己決定看哪裡

前置：向量、矩陣乘法和 softmax。先用單 head；每個小節都畫得出矩陣。完整模型仍留到第4章。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：問路譬喻：想找什麼、誰知道、拿回什麼**

Q/K 決定讀哪裡；V 是讀回的內容。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/qkv.svg")))

**先想一想：**兩個查詢、三個候選，分數矩陣是幾乘幾？

查詢 Q 表示「我在找什麼」，K 表示「我能被什麼需求找到」。兩組投影讓找的人與被找的人不必用同一種特徵。

**把直覺寫成數學：**$Q:[T_q,D_k]$，$K^T:[D_k,T_k]$，分數是 $[T_q,T_k]$；一列是一個 query。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
q = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
k = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
scores = q @ k.T
print("Q", q.shape, "K.T", k.T.shape, "scores", scores)
assert scores.shape == (2, 3)

**如何讀結果：**非方陣清楚顯示軸的角色；transpose 只換軸，不把 query 變成 key。

**只改一件事：**只增加一個 key，先預測輸出 shape。
